# Emotion Model Training

Train and evaluate the CNN used by the emojiFilter webcam app.

**Prerequisites**
- Install training dependencies: `pip install -r requirements-train.txt`
- Place labeled face images under `data/train/` — one folder per emotion
- Folder names must match `emoji_filter.config.EMOTIONS`
- Optional evaluation set at `data/test/` using the same layout

## Setup

In [ ]:
import logging
import sys
import time
from pathlib import Path

import numpy as np
import tensorflow as tf

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from emoji_filter.config import EMOTIONS
from emoji_filter.training.config import (
    EPOCHS_CNN,
    EPOCHS_TRANSFER,
    MODEL_EXPORT_PATH,
    TEST_DIR,
    TRAIN_DIR,
)
from emoji_filter.training.data import create_data_generators, summarize_data
from emoji_filter.training.models import build_cnn_model, build_mobilenet_model
from emoji_filter.training.visualization import (
    plot_augmented_samples,
    plot_confusion_matrix,
    plot_prediction_samples,
    plot_training_history,
)

tf.get_logger().setLevel(logging.ERROR)

## Data

In [ ]:
train_counts = summarize_data(TRAIN_DIR)
test_counts = summarize_data(TEST_DIR)

print('Training set:')
for label, count in train_counts.items():
    print(f'  {label}: {count}')
print(f"  total: {sum(train_counts.values())}")

print('\nTest set:')
for label, count in test_counts.items():
    print(f'  {label}: {count}')
print(f"  total: {sum(test_counts.values())}")

In [ ]:
train_data_gen, val_data_gen, test_data_gen = create_data_generators()

print('Class indices:', train_data_gen.class_indices)

In [ ]:
plot_augmented_samples(train_data_gen)

## CNN from scratch

In [ ]:
cnn_model = build_cnn_model(num_classes=len(EMOTIONS))
cnn_model.compile(
    optimizer='adam',
    loss=tf.keras.losses.CategoricalCrossentropy(),
    metrics=['accuracy'],
)
cnn_model.summary()

In [ ]:
start_time = time.time()

cnn_callbacks = [
    tf.keras.callbacks.EarlyStopping(patience=5, restore_best_weights=True),
]

cnn_history = cnn_model.fit(
    train_data_gen,
    epochs=EPOCHS_CNN,
    validation_data=val_data_gen,
    callbacks=cnn_callbacks,
)

print(f"Training finished in {(time.time() - start_time) / 60:.1f} minutes")

In [ ]:
plot_training_history(cnn_history, len(cnn_history.history['loss']), title_prefix='CNN')

## Evaluation

In [ ]:
def evaluate_model(model, generator):
    predictions = model.predict(generator)
    y_pred = np.argmax(predictions, axis=1)
    y_true = generator.classes
    return y_true, y_pred


y_true, y_pred = evaluate_model(cnn_model, test_data_gen)
plot_confusion_matrix(y_true, y_pred, EMOTIONS)

In [ ]:
images, labels = next(test_data_gen)
predictions = cnn_model.predict(images)
predicted_ids = np.argmax(predictions, axis=1)
label_ids = np.argmax(labels, axis=1)

plot_prediction_samples(images, label_ids, predicted_ids, EMOTIONS)

## Transfer learning (MobileNet V2)

In [ ]:
transfer_model = build_mobilenet_model(
    image_size=train_data_gen.target_size[0],
    num_classes=len(EMOTIONS),
    trainable=True,
)
transfer_model.compile(
    optimizer='adam',
    loss=tf.keras.losses.CategoricalCrossentropy(),
    metrics=['accuracy'],
)
transfer_model.summary()

In [ ]:
start_time = time.time()

transfer_history = transfer_model.fit(
    train_data_gen,
    epochs=EPOCHS_TRANSFER,
    validation_data=val_data_gen,
)

print(f"Training finished in {(time.time() - start_time) / 60:.1f} minutes")

In [ ]:
plot_training_history(
    transfer_history,
    len(transfer_history.history['loss']),
    title_prefix='MobileNet',
)

In [ ]:
y_true, y_pred = evaluate_model(transfer_model, test_data_gen)
plot_confusion_matrix(y_true, y_pred, EMOTIONS)

## Export

In [ ]:
MODEL_EXPORT_PATH.parent.mkdir(parents=True, exist_ok=True)
transfer_model.save(MODEL_EXPORT_PATH)
print(f'Saved model to {MODEL_EXPORT_PATH}')